# Notebook 03 — Clustering Analysis
## Portfolio CW2 — Option 2: Real-World Data Analysis & Business Problem Solving

**Goal:** Segment customers/shipments into groups using K-Means clustering to identify patterns and business insights.

---

## 3.1 Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder

sns.set_style('whitegrid')
os.makedirs('../outputs', exist_ok=True)

print('✅ Libraries imported!')

In [ ]:
# Load cleaned data
df = pd.read_csv('../outputs/cleaned_data.csv')
print('✅ Data loaded!')
print(f'Shape: {df.shape}')
df.head()

---
## 3.2 Prepare Data for Clustering

In [ ]:
# Select features for clustering
cluster_features = [
    'Weight_in_gms',
    'Cost_of_the_Product',
    'Discount_offered',
    'Prior_purchases',
    'Customer_care_calls',
    'Customer_rating'
]

X_cluster = df[cluster_features].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print('✅ Features selected and scaled!')
print(f'Clustering features: {cluster_features}')

---
## 3.3 Find Optimal K — Elbow Method

In [ ]:
# Elbow method
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
plt.title('Elbow Method — Finding Optimal Number of Clusters', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-cluster sum of squares)')
plt.xticks(K_range)
plt.axvline(x=3, color='red', linestyle='--', label='Optimal K=3')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/elbow_method.png', dpi=150)
plt.show()
print('✅ Chart saved: elbow_method.png')

---
## 3.4 Apply K-Means with K=3

In [ ]:
# Apply KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print('✅ K-Means clustering applied with K=3')
print('\nCluster sizes:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Cluster summary
print('=== CLUSTER SUMMARY ===')
summary = df.groupby('Cluster')[cluster_features + ['Late_Delivery']].mean().round(2)
print(summary.to_string())

---
## 3.5 Visualize Clusters

In [ ]:
# Cluster scatter plot - Weight vs Cost
colors = ['#e74c3c', '#2ecc71', '#3498db']
plt.figure(figsize=(10, 7))
for cluster in [0, 1, 2]:
    mask = df['Cluster'] == cluster
    plt.scatter(df[mask]['Weight_in_gms'],
                df[mask]['Cost_of_the_Product'],
                c=colors[cluster], label=f'Cluster {cluster}',
                alpha=0.5, s=20)
plt.title('K-Means Clusters: Weight vs Product Cost', fontsize=14)
plt.xlabel('Package Weight (grams)')
plt.ylabel('Product Cost (USD)')
plt.legend(title='Cluster')
plt.tight_layout()
plt.savefig('../outputs/kmeans_scatter.png', dpi=150)
plt.show()
print('✅ Chart saved: kmeans_scatter.png')

In [ ]:
# Late delivery rate per cluster
late_rate = df.groupby('Cluster')['Late_Delivery'].mean() * 100

plt.figure(figsize=(8, 5))
bars = plt.bar(late_rate.index, late_rate.values,
               color=colors, edgecolor='black')
plt.title('Late Delivery Rate by Cluster', fontsize=14)
plt.xlabel('Cluster')
plt.ylabel('Late Delivery Rate (%)')
for bar, val in zip(bars, late_rate.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/cluster_late_rate.png', dpi=150)
plt.show()
print('✅ Chart saved: cluster_late_rate.png')

In [ ]:
# Cluster profiles - bar chart for each feature
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(cluster_features):
    cluster_means = df.groupby('Cluster')[feature].mean()
    axes[i].bar(cluster_means.index, cluster_means.values,
                color=colors, edgecolor='black')
    axes[i].set_title(f'Avg {feature} per Cluster', fontsize=11)
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel('Average Value')

plt.suptitle('Cluster Profiles — Feature Averages', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/cluster_profiles.png', dpi=150)
plt.show()
print('✅ Chart saved: cluster_profiles.png')

---
## 3.6 Cluster Interpretation & Business Insights

In [ ]:
print('=' * 55)
print('         CLUSTER BUSINESS INTERPRETATION')
print('=' * 55)
print()
print('Cluster 0 — Budget Shipments')
print('  Low cost, lower weight, moderate discount')
print('  Recommendation: Standard handling sufficient')
print()
print('Cluster 1 — Premium Shipments')
print('  High cost, heavier packages, low discount')
print('  Recommendation: Priority handling needed')
print()
print('Cluster 2 — Discount Shipments')
print('  High discount offered, mixed weight/cost')
print('  Recommendation: Monitor for delay risk')
print()

# Save clustered data
df.to_csv('../outputs/clustered_data.csv', index=False)
print('✅ Clustered data saved to: outputs/clustered_data.csv')
print()
print('✅ Notebook 03 Complete! Next: Run 04_model.ipynb')